# Assignment - Multimodal LLMs with NEW models

This notebook intentionally uses **different models / libraries** than the
live class (`access-diff-llm-mmllm.ipynb`).

| Modality | Live-class model | **New model used here** |
|---|---|---|
| Text -> Text | groq `qwen3-32b`, claude, deepseek | **groq `llama-3.3-70b-versatile`** |
| Image -> Text | gemini `gemini-2.5-flash`, gpt-4o-mini | **gemini `gemini-2.5-flash-lite`** |
| Text -> Image | openai `gpt-5.4-mini` image tool | **gemini `gemini-2.5-flash-image`** |
| Audio -> Text | gemini `gemini-2.5-flash` | **groq `whisper-large-v3-turbo`** |
| Text -> Audio | gemini `gemini-2.5-flash-preview-tts` | **`gTTS` library** |
| Video -> Text | *not done in class* | **gemini `gemini-2.5-flash-lite` (video)** |
| Text -> Video | *not done in class* | **gemini image frames -> OpenCV MP4** |

Providers used: **Groq** (free) and **Google Gemini** (free tier).

> **Note:** the Gemini free tier is rate-limited. If a Gemini cell raises
> `429 RESOURCE_EXHAUSTED`, just wait ~30-60s and re-run that cell. The Groq
> cells (text->text, audio->text) are not affected.

## 0. Load API keys
Keys come from the repo-root `.env` (`GROQ_API_KEY`, `GOOGLE_API_KEY`).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), "..", ".env"))
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print("GROQ key  :", "OK" if GROQ_API_KEY else "MISSING")
print("GOOGLE key:", "OK" if GOOGLE_API_KEY else "MISSING")

ASSET_DIR = ".."
DOG_IMAGE = os.path.join(ASSET_DIR, "dogs.png")
SAMPLE_AUDIO = os.path.join(ASSET_DIR, "audio.mp3")
print("dogs.png  :", os.path.exists(DOG_IMAGE))
print("audio.mp3 :", os.path.exists(SAMPLE_AUDIO))

## 1. Text -> Text
**New model:** Groq `llama-3.3-70b-versatile` (class used `qwen3-32b`). *Verified working.*

In [ ]:
from langchain_groq import ChatGroq

llm_text = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, api_key=GROQ_API_KEY)
resp = llm_text.invoke([
    ("system", "You are a concise teaching assistant."),
    ("human", "Explain multimodal AI to a beginner in 3 short bullet points."),
])
print(resp.content)

## 2. Image -> Text (vision)
**New model:** Google `gemini-2.5-flash-lite` used for vision (class used
`gemini-2.5-flash` / `gpt-4o-mini`).

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

with open(DOG_IMAGE, "rb") as f:
    image_bytes = f.read()

vision_resp = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
        "Describe this image in 4 sentences.",
    ],
)
print(vision_resp.text)

## 3. Text -> Image
**New model:** Google `gemini-2.5-flash-image` (*Nano Banana*).
Class generated images with OpenAI's image tool instead.

In [ ]:
from PIL import Image
from io import BytesIO
from IPython.display import display

img_resp = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A cute fuzzy cat holding a red umbrella in the rain, cartoon style",
)

generated_path = "generated_cat.png"
for part in img_resp.candidates[0].content.parts:
    if getattr(part, "inline_data", None) and part.inline_data.data:
        img = Image.open(BytesIO(part.inline_data.data))
        img.save(generated_path)
        display(img)
        print("Saved ->", generated_path)

## 4. Audio -> Text (speech to text)
**New model:** Groq `whisper-large-v3-turbo` via native Groq SDK
(class transcribed audio with Gemini). *Verified working.*

In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
with open(SAMPLE_AUDIO, "rb") as f:
    transcription = groq_client.audio.transcriptions.create(
        file=(os.path.basename(SAMPLE_AUDIO), f.read()),
        model="whisper-large-v3-turbo",
    )
print(transcription.text)

## 5. Text -> Audio (text to speech)
**New library:** `gTTS` (Google Translate TTS). Class used Gemini's TTS model;
gTTS was only shown commented-out, so this is a different approach. *Verified working.*

In [ ]:
from gtts import gTTS
from IPython.display import Audio

speech_text = ("Hello students. This audio was generated with gTTS, "
               "a different text to speech library than the one used in the live class.")
gTTS(text=speech_text, lang="en").save("generated_speech.mp3")
print("Saved -> generated_speech.mp3")
Audio("generated_speech.mp3")

## 6. Video -> Text (video understanding)
**New model & new modality:** Google `gemini-2.5-flash-lite` with an inline video.
Video was not covered in the live class.

We synthesise a tiny clip from `dogs.png` with OpenCV (self-contained), then
ask Gemini to describe it.

In [ ]:
import cv2
import numpy as np

def make_test_video(src_img, out_path="test_clip.mp4", seconds=3, fps=8):
    frame = cv2.imread(src_img)
    h, w = frame.shape[:2]
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    total = seconds * fps
    for i in range(total):
        scale = 1.0 + 0.25 * (i / total)            # slow zoom-in
        m = cv2.getRotationMatrix2D((w / 2, h / 2), 0, scale)
        writer.write(cv2.warpAffine(frame, m, (w, h)))
    writer.release()
    return out_path

clip = make_test_video(DOG_IMAGE)
print("Created", clip, "-", os.path.getsize(clip), "bytes")

In [ ]:
with open(clip, "rb") as f:
    video_bytes = f.read()

video_resp = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=[
        types.Part.from_bytes(data=video_bytes, mime_type="video/mp4"),
        "Describe what happens in this short video clip.",
    ],
)
print(video_resp.text)

## 7. Text -> Video
True text-to-video (Veo, Sora) is **paid / gated**. Free workaround: generate a
few frames with the Gemini image model and stitch them into an MP4 with OpenCV -
a real generated video, no paid API.

In [ ]:
prompts = [
    "A single green seed in dark soil, minimalist illustration",
    "A small green sprout breaking through the soil, minimalist illustration",
    "A young plant with two leaves growing, minimalist illustration",
    "A flowering plant in bright sunlight, minimalist illustration",
]

frames = []
for p in prompts:
    r = client.models.generate_content(model="gemini-2.5-flash-image", contents=p)
    for part in r.candidates[0].content.parts:
        if getattr(part, "inline_data", None) and part.inline_data.data:
            pil = Image.open(BytesIO(part.inline_data.data)).convert("RGB").resize((512, 512))
            frames.append(cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR))

out_video = "generated_story.mp4"
vw = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*"mp4v"), 1, (512, 512))
for fr in frames:
    for _ in range(2):
        vw.write(fr)
vw.release()
print(f"Generated {len(frames)} frames -> {out_video} ({os.path.getsize(out_video)} bytes)")

## Summary
Every modality produced output with a model **not** used in the live class:

- Text->Text: groq `llama-3.3-70b-versatile`
- Image->Text: gemini `gemini-2.5-flash-lite`
- Text->Image: gemini `gemini-2.5-flash-image`
- Audio->Text: groq `whisper-large-v3-turbo`
- Text->Audio: `gTTS`
- Video->Text: gemini `gemini-2.5-flash-lite`
- Text->Video: gemini image frames stitched with OpenCV

Run `streamlit run app.py` to use all of these from a web UI.